# MS MARCO v2 embeddings on Colab (e5-base-v2 + PCA 192, float16)

Embeds the 138,364,198 MS MARCO v2 passages in **48 shards** (~2.9M each) straight from the raw `.gz` files (no
58 GB `corpus.jsonl` needed), and copies every finished shard to Google Drive. The rows, ids and text are exactly
what `prepare_dataset.py` + `collection_preproc.py` produce on the server, so the server can use them directly.

**Runtime:** *Runtime → Change runtime type → L4 GPU* (cheapest per passage; A100 is ~2x faster for a bit more).

**Download once:** run the first cells (up to the passage count) on a cheap *CPU* runtime first: it puts the 20 GB
of `.gz` files and their counts on Drive, so the GPU tabs skip straight to embedding. **Resumable:** if a session
dies, just *Run all* again: it skips shards already on Drive. **Faster:** open this notebook in up to 6 tabs (each tab is its own runtime + GPU) with `NUM_SESSIONS = 6`
and `SESSION = 0..5`; each does every 6th shard. SESSION 0 trains the PCA, the others wait for it on Drive.

When all 48 are on Drive: copy the Drive folder to the server's `datasets/msmarco-v2/local_import/` and run
`bash datasets/import_local_embeds.sh msmarco-v2` there (it merges them into float32 and embeds the queries).

In [ ]:
# ---- settings
DRIVE_DIR    = "/content/drive/MyDrive/pillar/msmarco-v2"   # shards + PCA end up here (~55 GB)
NUM_SHARDS   = 48
SESSION, NUM_SESSIONS = 0, 1        # 6 tabs: NUM_SESSIONS = 6 and SESSION = 0..5 (one per tab)
ENCODE_BATCH = 256                  # L4 / A100 / T4 all fit 256
REPO   = "https://github.com/dkblackley/bins-go.git"   # private repo? add a GITHUB_TOKEN in Colab's Secrets (key icon)
MARCO  = "https://msmarco.z22.web.core.windows.net/msmarcoranking/msmarco_v2_passage.tar"
EXPECTED_PASSAGES = 138_364_198

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess, shutil, glob, json, time
os.makedirs(DRIVE_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

def run(cmd):
    """subprocess whose output (prints, tqdm bars) shows up in the cell; Colab hides plain subprocess output"""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    for chunk in iter(lambda: p.stdout.read(256), b""):
        sys.stdout.write(chunk.decode("utf-8", "replace")); sys.stdout.flush()
    if p.wait():
        raise RuntimeError(f"{cmd[:3]} ... exited with {p.returncode}")


In [ ]:
# ---- code + libraries (Colab already has torch)
if not os.path.exists("/content/bins-go"):
    url = REPO
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        url = REPO.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/bins-go"], check=True)
!pip -q install sentence-transformers faiss-cpu

In [ ]:
# ---- MS MARCO v2 passages: streamed straight out of the tar into Drive (~20 min, only the first session does it;
# every later session/tab reads the .gz files from Drive, each shard only opens the 1-2 files it needs)
RAW = f"{DRIVE_DIR}/raw/msmarco_v2_passage"
if len(glob.glob(f"{RAW}/msmarco_passage_*.gz")) < 70 or not os.path.exists(f"{RAW}/msmarco_passage_69.gz"):
    import requests, tarfile
    from tqdm.auto import tqdm
    os.makedirs(f"{DRIVE_DIR}/raw", exist_ok=True)
    t = time.time()
    r = requests.get(MARCO, stream=True, timeout=60); r.raise_for_status()
    bar = tqdm(total=int(r.headers.get("content-length", 0)) or None, unit="B", unit_scale=True, desc="download + untar")
    class Counted:   # file-like wrapper that moves the bar as tarfile reads the stream
        def read(self, k):
            b = r.raw.read(k); bar.update(len(b)); return b
    with tarfile.open(fileobj=Counted(), mode="r|") as tf:
        tf.extractall(f"{DRIVE_DIR}/raw")
    bar.close()
    print(f"downloaded in {(time.time() - t) / 60:.1f} min")
# passage counts per file: computed once (~3 min), kept on Drive for the next session
if os.path.exists(f"{DRIVE_DIR}/line_counts.json") and not os.path.exists(f"{RAW}/line_counts.json"):
    shutil.copy(f"{DRIVE_DIR}/line_counts.json", f"{RAW}/line_counts.json")
os.chdir("/content/bins-go")
import collection_preproc as cp
n = sum(cp.raw_line_counts(cp.raw_msmarco_v2_files(RAW)))
shutil.copy(f"{RAW}/line_counts.json", f"{DRIVE_DIR}/line_counts.json")
print(f"{n:,} passages")
assert n == EXPECTED_PASSAGES, "passage count differs from the server's corpus, don't use these"

In [ ]:
# ---- PCA (768 -> 192): trained once on the first 100k passages (same sample as the server would use) by SESSION 0
# only, and shared with every shard and every other session through Drive. Other sessions wait for it, so all
# shards use the same PCA.
OUT = "/content/out"; os.makedirs(OUT, exist_ok=True)
PCA = f"{OUT}/pca_e5_768_to_192.faiss"
EMB = f"{OUT}/msmarco-v2_corpus_embed_f16.npy"
DRIVE_PCA = f"{DRIVE_DIR}/pca_e5_768_to_192.faiss"
if not os.path.exists(DRIVE_PCA) and SESSION == 0:
    run(["python", "-u", "collection_preproc.py", "--documents", RAW, "--output", EMB, "--pca", PCA, "--pca-only"])
    shutil.copy(PCA, DRIVE_PCA + ".uploading"); os.replace(DRIVE_PCA + ".uploading", DRIVE_PCA)
while not os.path.exists(DRIVE_PCA):
    print("waiting for SESSION 0 to put the PCA on Drive ..."); time.sleep(30)
shutil.copy(DRIVE_PCA, PCA)
print("PCA ready")


In [ ]:
# ---- the shards: embed on local disk, copy to Drive (temp name + rename), delete locally
def on_drive(i):
    s = f"{DRIVE_DIR}/msmarco-v2_corpus_embed_f16.npy.shard{i:03d}of{NUM_SHARDS:03d}.npy"
    return os.path.exists(s) and os.path.exists(s + ".ids")

mine = [i for i in range(NUM_SHARDS) if i % NUM_SESSIONS == SESSION]
t0 = time.time(); todo = [i for i in mine if not on_drive(i)]
print(f"this session: shards {mine}, {len(todo)} to do")
for k, i in enumerate(mine):
    if on_drive(i):
        print(f"shard {i}: already on Drive"); continue
    t = time.time()
    run(["python", "-u", "collection_preproc.py", "--documents", RAW, "--output", EMB, "--pca", PCA,
                    "--load-pca", "--num-docs", str(EXPECTED_PASSAGES), "--num-shards", str(NUM_SHARDS),
                    "--shard-id", str(i), "--dtype", "float16", "--fp16", "--encode-batch", str(ENCODE_BATCH),
                    "--skip-if-done"])
    local = f"{EMB}.shard{i:03d}of{NUM_SHARDS:03d}.npy"
    dst = f"{DRIVE_DIR}/{os.path.basename(local)}"
    for src, d in ((local + ".ids", dst + ".ids"), (local, dst)):   # .npy last: it marks the shard as done
        shutil.copy(src, d + ".uploading"); os.replace(d + ".uploading", d)
    for f in glob.glob(local + "*"):
        os.remove(f)
    n_done = sum(1 for j in todo if on_drive(j))
    left = (time.time() - t0) / max(n_done, 1) * (len(todo) - n_done)
    print(f"shard {i}: done in {(time.time() - t) / 60:.1f} min | session {n_done}/{len(todo)}, ETA {left / 60:.0f} min")

In [ ]:
# ---- status
done = [i for i in range(NUM_SHARDS) if on_drive(i)]
print(f"{len(done)}/{NUM_SHARDS} shards on Drive; missing: {[i for i in range(NUM_SHARDS) if i not in done]}")
if len(done) == NUM_SHARDS:
    print("All done. Copy", DRIVE_DIR, "-> <server>/datasets/msmarco-v2/local_import/ (the shard .npy + .ids files")
    print("and pca_e5_768_to_192.faiss), then on the server: bash datasets/import_local_embeds.sh msmarco-v2")

## Getting the files onto the server with plain `wget`

Run cells 1-2, then the cell below (a CPU runtime is enough). It writes `download_msmarco_v2.sh` into the Drive folder: one `wget` per file (PCA + 48 shards + their `.ids`), using each file's Drive id. It then prints a single command to paste on the server. The folder must be shared as *Anyone with the link* (files in it inherit that). The script skips files that are already there with the right size, so if Drive rate-limits a download, just run it again.

In [ ]:
# ---- one-command download for the server: a wget per file, straight from Drive (no zip, no extra Drive space)
import shlex
files = ["pca_e5_768_to_192.faiss"]
for i in range(NUM_SHARDS):
    s = f"msmarco-v2_corpus_embed_f16.npy.shard{i:03d}of{NUM_SHARDS:03d}.npy"
    files += [s, s + ".ids"]
missing = [f for f in files if not os.path.exists(f"{DRIVE_DIR}/{f}")]
assert not missing, f"{len(missing)} files not on Drive yet, e.g. {missing[:3]}"

def drive_id(path, wait=600):
    """Drive file id from the Colab Drive mount. A file that hasn't been uploaded yet has a placeholder id
    ("local-123") that Drive's servers don't know, so wait until the real one shows up."""
    for k in range(wait // 10):
        try:
            i = os.getxattr(path, "user.drive.id").decode()
            if not i.startswith("local-"):
                return i
        except OSError:
            pass
        if k == 0:
            print(f"waiting for {os.path.basename(path)} to finish uploading to Drive ...")
        os.sync(); time.sleep(10)
    raise RuntimeError(f"no Drive id for {path} after {wait} s: run this cell again, or open the file in Drive, "
                       f"Share -> Copy link, and use the part after /d/ as the id")

script = [
    "#!/bin/bash",
    "# Written by datasets/colab_msmarco_v2.ipynb. Run it in the folder the files should go to, e.g.",
    "#   mkdir -p ../datasets/msmarco-v2/local_import && cd ../datasets/msmarco-v2/local_import && bash download_msmarco_v2.sh",
    "# Re-run to retry: files that are already here with the right size are skipped.",
    "bad=0",
    "get() {   # <drive id> <file name> <bytes>",
    '    if [[ -f "$2" && $(stat -c %s "$2") == "$3" ]]; then echo "ok      $2"; return; fi',
    '    wget -q --show-progress -O "$2.part" "https://drive.usercontent.google.com/download?id=$1&export=download&confirm=t"',
    '    local got; got=$(stat -c %s "$2.part" 2>/dev/null || echo 0)',
    '    if [[ "$got" == "$3" ]]; then mv "$2.part" "$2"; echo "got     $2"; else',
    '        echo "FAILED  $2: $got of $3 bytes ($(head -c 300 "$2.part" 2>/dev/null | tr -cd "[:print:]" | cut -c1-150))" >&2',
    '        rm -f "$2.part"; bad=1; fi',
    "}",
]
for f in files:
    p = f"{DRIVE_DIR}/{f}"
    script.append(f"get {drive_id(p)} {shlex.quote(f)} {os.path.getsize(p)}")
script += [
    f'if [[ $bad == 0 ]]; then echo "all {len(files)} files are here"; else',
    '    echo "some files failed; run the script again (if Drive says quota exceeded, wait a few hours)" >&2; exit 1; fi',
]
SH = f"{DRIVE_DIR}/download_msmarco_v2.sh"
with open(SH, "w") as f:
    f.write("\n".join(script) + "\n")
drive.flush_and_unmount(); drive.mount("/content/drive")   # pushes the new file to Drive now instead of "eventually"
url = f"https://drive.usercontent.google.com/download?id={drive_id(SH)}&export=download&confirm=t"
print("On the server, from the bins-go repo root (in tmux, ~55 GB):\n")
print("mkdir -p ../datasets/msmarco-v2/local_import && cd ../datasets/msmarco-v2/local_import && "
      f'wget -O download_msmarco_v2.sh "{url}" && bash download_msmarco_v2.sh')